# Model_Test · External news classification

Use the frozen Naive Bayes pipeline from Notebook 3 to classify the imported external news sample in `data/external/Test_News/external_test_news.csv`. Compare predictions with the dataset labels as a narrow external evaluation. This notebook only loads and applies the saved model: **it does not retrain, tune, or alter the holdout results**.

The dataset contains cached English COVID-19 articles with inherited CrossFake labels. Its 95 articles are imbalanced (20 fake, 75 real), and its topic and source distribution are narrow. Results should be interpreted as a supplementary test on this dataset, not as a representative measure of performance on all current world news or as fact-checking.


## 1. Load the model and external data

The model was trained on `title + "\n\n" + text`; reproduce that exact input construction. Provenance columns are for audit and are not passed into the classifier.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib.metadata as package_metadata
import json
import platform

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data' / 'raw' / 'News').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from data/raw/News.')

data_path = PROJECT_ROOT / 'data' / 'external' / 'Test_News' / 'external_test_news.csv'
model_path = PROJECT_ROOT / 'models' / 'final_naive_bayes_pipeline.joblib'
for path in [data_path, model_path]:
    if not path.exists():
        raise FileNotFoundError(f'Required data/model artifact is missing: {path}')

external = pd.read_csv(data_path, keep_default_na=False)
required_columns = {'article_id', 'title', 'text', 'label'}
missing_columns = required_columns - set(external.columns)
if missing_columns:
    raise ValueError(f'External data is missing required columns: {sorted(missing_columns)}')
if external.empty or external['article_id'].duplicated().any():
    raise ValueError('External data must contain rows with unique article IDs.')
if not set(external['label']).issubset({'fake', 'real'}):
    raise ValueError('Labels must be standardized to `fake` and `real`.')
if external[['title', 'text']].eq('').all(axis=1).any():
    raise ValueError('An article has both an empty title and empty body.')

external['model_text'] = external['title'] + '\n\n' + external['text']
print(f'External articles: {len(external):,}')
print('Dataset label counts:')
display(external['label'].value_counts().rename_axis('label').to_frame('articles'))
if 'source_type' in external.columns:
    print('Source type counts by dataset label (audit only; not model input):')
    display(pd.crosstab(external['label'], external['source_type']))
print('Loaded frozen model:', model_path)


## 2. Classify the external articles

Each row is passed through the already-fitted vectorizer and Naive Bayes classifier. The `label` column is retained only for evaluation; it is never used as input to prediction. The display below is a reproducible random sample of the classified rows; the metrics and saved predictions cover the full external dataset.


In [ ]:
pipeline = joblib.load(model_path)
y_true = external['label'].to_numpy()
y_pred = pipeline.predict(external['model_text']).astype(str)
labels = ['fake', 'real']
if set(np.unique(y_true)) != set(labels):
    raise ValueError(f'Expected both dataset labels {labels}; observed {sorted(np.unique(y_true))}.')
if not set(np.unique(y_pred)).issubset(labels):
    raise ValueError(f'Model returned unexpected labels: {sorted(np.unique(y_pred))}.')

predictions = external[['article_id', 'title', 'label']].copy()
predictions = predictions.rename(columns={'label': 'actual_label'})
predictions['predicted_label'] = y_pred
predictions['correct'] = predictions['actual_label'].to_numpy() == y_pred
RANDOM_SEED = 42
DISPLAY_SAMPLE_SIZE = 20
random_prediction_sample = predictions.sample(
    n=min(DISPLAY_SAMPLE_SIZE, len(predictions)), random_state=RANDOM_SEED
).reset_index(drop=True)
print(f'Reproducible random sample of {len(random_prediction_sample)} classified articles (seed={RANDOM_SEED}):')
display(random_prediction_sample[['article_id', 'title', 'actual_label', 'predicted_label', 'correct']])


## 3. Evaluate against the provided labels

Macro F1 is the primary summary metric, with accuracy, balanced accuracy, macro precision/recall, and per-class results for context. Because the sample has many more real than fake articles, accuracy alone can hide weaker performance on the fake class.


In [ ]:
metrics = {
    'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    'evaluation_set': 'CrossFake English COVID-19 articles; external to project training corpus by exact normalized title/body check',
    'n_articles': int(len(y_true)),
    'label_counts': {str(k): int(v) for k, v in external['label'].value_counts().items()},
    'macro_f1_primary': float(f1_score(y_true, y_pred, labels=labels, average='macro')),
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    'macro_precision': float(precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    'macro_recall': float(recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    'model_artifact': str(model_path.relative_to(PROJECT_ROOT)),
    'dataset_path': str(data_path.relative_to(PROJECT_ROOT)),
    'python_version': platform.python_version(),
    'scikit_learn_version': package_metadata.version('scikit-learn'),
}
summary = pd.DataFrame([
    {'metric': 'Macro F1 (primary)', 'value': metrics['macro_f1_primary']},
    {'metric': 'Accuracy', 'value': metrics['accuracy']},
    {'metric': 'Balanced accuracy', 'value': metrics['balanced_accuracy']},
    {'metric': 'Macro precision', 'value': metrics['macro_precision']},
    {'metric': 'Macro recall', 'value': metrics['macro_recall']},
    {'metric': 'Articles', 'value': metrics['n_articles']},
])
display(summary.round(4))

report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
report_table = pd.DataFrame(report).T
display(report_table.round(4))


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted label')
ax.set_ylabel('Dataset label')
ax.set_title('External sample confusion matrix', loc='left', pad=12)
fig.tight_layout()
plt.show()

print('Misclassified articles:')
misclassified = predictions.loc[~predictions['correct']].copy()
display(misclassified[['article_id', 'title', 'actual_label', 'predicted_label']])
print(f'Errors: {len(misclassified):,} / {len(predictions):,}')


## 4. Save predictions and evaluation metrics

The predictions file includes IDs and titles for interpretation, actual labels, model predictions, and correctness. The aggregate JSON captures the external metrics and evaluation context. No article body is duplicated in these result files.


In [ ]:
output_dir = PROJECT_ROOT / 'outputs' / 'external_test'
output_dir.mkdir(parents=True, exist_ok=True)
predictions_path = output_dir / 'external_test_predictions.csv'
metrics_path = output_dir / 'external_test_metrics.json'

predictions.to_csv(predictions_path, index=False, encoding='utf-8')
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print('Saved predictions:', predictions_path)
print('Saved metrics:', metrics_path)


## 5. Interpretation and limitations

After execution, record macro F1, balanced accuracy, and fake/real precision, recall, and F1. Inspect the confusion matrix and misclassified items.

These results apply only to this small, imbalanced, COVID-19-focused sample and to its inherited labels. They do not show that the model can fact-check arbitrary claims or generalize to present-day world news. The sample's source mix differs across labels (shown above), which can create source-style shortcuts. Do not tune the model using these examples and then reuse their score as an independent test. Any individual prediction is a text-pattern classification, not a determination of truth.
